<a href="https://colab.research.google.com/github/AndreaaCosentino/QMC/blob/main/QMC_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
using Pkg
import CUDA
using Pkg
using QuantumOptics
using Arpack
using LinearAlgebra
using Test
using Combinatorics
using JSON3
using StructTypes
using KrylovKit
import Base: isless
using Base.Threads
using CUDA
using OrdinaryDiffEq
using OrdinaryDiffEqLowOrderRK
using CUDA.CUSOLVER

In [7]:
Pkg.add(["QuantumOptics", "Arpack", "Plots", "Combinatorics", "JSON3", "StructTypes", "KrylovKit","OrdinaryDiffEq","OrdinaryDiffEqLowOrderRK"]);


    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed TimerOutputs ────────────────── v0.5.29
   Installed NonlinearSolve ──────────────── v4.19.1
   Installed Accessors ───────────────────── v0.1.44
   Installed StochasticDiffEqRODE ────────── v2.0.0
   Installed SciMLLogging ────────────────── v2.0.0
   Installed LRUCache ────────────────────── v1.6.2
   Installed FFTW ────────────────────────── v1.10.0
   Installed FastGaussQuadrature ─────────── v1.2.0
   Installed FunctionWrappers ────────────── v1.1.3
   Installed IntelOpenMP_jll ─────────────── v2025.2.0+0
   Installed MaybeInplace ────────────────── v0.1.4
   Installed RationalRoots ───────────────── v0.2.1
   Installed OrdinaryDiffEqCore ──────────── v4.3.0
   Installed SciMLJacobianOperators ──────── v0.1.13
   Installed TupleTools ──────────────────── v1.6.0
   Installed HalfIntegers ────────────────── v1.6.0
   Installed FastPower ───────────────────── v1.3.1
   Installed 

In [2]:
function tuple_to_mask(t)
    m = 0
    for i in t
        m |= (1 << (i - 1))
    end
    return m
end

const b = SpinBasis(1/2)
const z = sigmaz(b)
const y = sigmay(b)
const x = sigmax(b)
const i = identityoperator(b);

function ground_state(m, H, B)
    _, vecs, _ = eigsolve(H, 1, :SR)
    v = vecs[1]
    return Ket(B, CuArray(complex(v)))
end;

In [3]:
struct LWGraph
    nodes::Vector{Int64}
end

function LWinit_graph(n)
    nodes = zeros(n)
    return LWGraph(nodes)
end

function LWadd_edge(G,i,j)
    G.nodes[i] = G.nodes[i] | 1 << (j-1)
    G.nodes[j] = G.nodes[j] | 1 << (i-1)
end

function LWdegree(G,i)
    return count_ones(G.nodes[i])
end

function LWedge_exist(G,i,j)
    return (G.nodes[i] & 1 << (j-1)) != 0
end

function LWget_pos(bitstring)
    positions = Int[]

    while bitstring > 0

        pos = trailing_zeros(bitstring) + 1
        push!(positions, pos)

        bitstring = bitstring & (bitstring - 1)
    end

    return positions
end

function LWnodes(n,k)
    c = combinations(1:n,k)
    e = []

    for i in c
        s = tuple_to_mask(i)
        push!(e,s)
    end
    return e
end

function LWtoken_graph(G::LWGraph)
    n = size(G.nodes)[1]
    k = Int(floor(n/2))
    e = LWnodes(n,k)

    i = binomial(n, k)

    Gt = LWinit_graph(i)

    for j = 1:i
        l = 0
        for k = j+1:i
            s = xor(e[k],e[j])

            count_ones(s) == 2 || continue

            v1 = trailing_zeros(s) + 1
            v2 = trailing_zeros(s & (s - 1)) + 1

            if LWedge_exist(G,v1,v2)
                LWadd_edge(Gt,j,k)
            end
        end
    end

    return Gt
end

function LW_laplacian(G)
    n = size(G.nodes)[1]
    L = zeros(n,n)
    for i in 1:n
        for j in 1:n
            if i == j
                L[i,j] = LWdegree(G,i)
            elseif LWedge_exist(G,i,j)
                L[i,j] = -1
            end
        end
    end
    return L
end
;

In [4]:
function LWinitial_Hamiltonian(n,G)
    B = b^n
    k = Int(floor(n/2))
    nodes = G.nodes

    term_0 = - LazySum([
        0.25 * (one(B) +  LazyTensor(B, [i,j], (z, z)) +  LazyTensor(B, [i], (z,)) + LazyTensor(B, [j], (z,)))
        for i in 1:n for j in LWget_pos(G.nodes[i]) if i < j
    ]...)

    m = nextpow(2,binomial(n,k))

    A = zeros(m,2^n)

    c = LWnodes(n,k)

    j = 1
    @views for e in c
        A[j,:] = dense(term_0).data[e,:]
        j += 1
    end

    B = zeros(m,m)
    j = 1
    @views for e in c
        B[:,j] = A[:,e]
        j += 1
    end

    s = Int(log2(m))
    H_0 = Operator(b^s,b^s,complex(B))
    return H_0
end;

function LWgenerate_graph(n)
    G = LWinit_graph(n)

    e = rand(1:n*(n-1)/2)
    for l = 1:e
        while true
            i = rand(1:n)
            j = 0
            while true
                j = rand(1:n)
                if i != j
                    break
                end
            end

            if i > j
                i,j = j,i
            end

            if !LWedge_exist(G,i,j)
                LWadd_edge(G,i,j)
                break
            end
        end
    end
    return G
end;

function LWrandom_Hamiltonian(n)
    k = Int(floor(n/2))

    G = LWgenerate_graph(n)
    L = LW_laplacian(LWtoken_graph(G))

    m = nextpow(2,binomial(n,k))
    s = Int(log2(m))

    H_f = zeros(m,m)
    H_f[1:binomial(n,k),1:binomial(n,k)] = -L

    H_f = Operator(b^s,b^s,H_f)
    return H_f,G
end;

function LWcorona_Hamiltonian(n)
    k = Int(floor(n/2))
    G = LWinit_graph(n)

    for i in 1:n-k
        for j in i+1:n-k
            LWadd_edge(G,i,j)
        end
    end

    for i in n-k+1:n
        LWadd_edge(G,i,i%(n-k)+1)
    end

    L = LW_laplacian(LWtoken_graph(G))

    m = nextpow(2,binomial(n,k))
    s = Int(log2(m))

    H_f = zeros(m,m)
    H_f[1:binomial(n,k),1:binomial(n,k)] = -L

    H_f = Operator(b^s,b^s,H_f)
    return H_f,G
end;

In [21]:
function LWhybrid_simulation(n)
    @time H_f,G = LWrandom_Hamiltonian(n)
    @time H_0,_ = LWcorona_Hamiltonian(n)

    k = Int(floor(n/2))
    m = nextpow(2,binomial(n,k))
    s = Int(log2(m))

    ground = ground_state(m,real(dense(H_0).data),b^s)

    T = 10
    dt = 0.01

    @time res = sim(T,dt,H_0,H_f,ground)

    psi_end = res[end].data
    H_end_gpu = CuArray{ComplexF64}(dense(H_f).data)
    H_end_cpu = dense(H_f).data
    psi_end_cpu = Array(psi_end)

    gpu_val = real(dot(psi_end, H_end_gpu * psi_end))
    cpu_val = real(dot(psi_end_cpu, H_end_cpu * psi_end_cpu))

    @time  v = plot_exp_value_gpu(H_0,H_f,res,dt,T)


    H_f_matrix = Hermitian(Array(dense(H_f).data))
    the_ground_eigenvalue = eigs(H_f_matrix, nev=1, which=:SR)[1][1]

    p = v / the_ground_eigenvalue

    return G,p
end;

In [6]:
function sim(T, dt, term_0, term_1, psi_gpu)
    u0 = psi_gpu.data
    tspan = (0.0, Float64(T))

    d0 = CuArray(dense(term_0).data)
    d1 = CuArray(dense(term_1).data)

    H_buf = copy(d0)

    f_schroedinger! = let d0=d0, d1=d1, H_buf=H_buf, T=T
        function(du, u, p, t)
            s = t / T
            @. H_buf = (1.0 - s) * d0 + s * d1
            mul!(du, H_buf, u, -im, 0.0)
        end
    end

    prob = ODEProblem(f_schroedinger!, u0, tspan)

    saveat_times = isnothing(dt) ? (0.0:T) : (0.0:dt:T)

    sol = solve(prob, Tsit5(); saveat=saveat_times, abstol=1e-6, reltol=1e-6)


    res = [Ket(psi_gpu.basis, u) for u in sol.u]

    return res
end

sim (generic function with 1 method)

In [33]:
LWhybrid_simulation(12)

  0.044339 seconds (390.07 k allocations: 27.083 MiB, 10.83% gc time)
  0.040614 seconds (390.07 k allocations: 27.083 MiB, 10.28% gc time)
  1.259065 seconds (429.92 k allocations: 30.031 MiB, 5.91% gc time, 1 lock conflict)
  0.441960 seconds (359.62 k allocations: 55.701 MiB, 1 lock conflict)


(LWGraph([1964, 3516, 2691, 3603, 2186, 195, 2208, 375, 1667, 1293, 779, 94]), 0.8852695268502384)

In [7]:
function plot_exp_value_gpu(term_0, term_1, psi, dt, T)
    s_vals = 0.0:Float64(dt):T
    energies = zeros(length(s_vals))
    d0 = CuArray{ComplexF64}(dense(term_0).data)

    d1 = CuArray{ComplexF64}(dense(term_1).data)
    H_buf = copy(d0)
    basis_l = term_0.basis_l
    basis_r = term_0.basis_r

    for (i, s) in enumerate(s_vals)
        param = s / T
        @. H_buf = (1 - param) * d0 + param * d1
        H_op = Operator(basis_l, basis_r, H_buf)
        psi_gpu = psi[i].data
        energies[i] = real(dot(psi_gpu, H_buf * psi_gpu))
    end

    return energies[end]
end

plot_exp_value_gpu (generic function with 1 method)